# CDF and Community Projects Dataset - Kalomo Town Council

**Owner:** Louis | **CSC4792 Mini Project, Group 44 - Kalomo Town Council, Zambia**

This notebook documents how `db-unza26-csc4792-kalomo_town_council_cdf_projects.csv` was produced from scanned Constituency Development Fund (CDF) project lists published by Kalomo Town Council. It covers source selection, OCR extraction, table reconstruction, cleaning decisions, validation, and the limitations of the final dataset.

The supporting scripts are:

- `scripts/scraping/scrape_cdf_projects.py`
- `scripts/cleaning/clean_cdf_projects.py`

The expensive OCR stage was run in Google Colab because EasyOCR benefits from a GPU. The notebook uses the saved raw and processed CSV files, so it can be reviewed without downloading the PDFs or running OCR again.

## Step 1 - Identify and select the source documents

The council website provided 20 CDF-related PDFs. They were reviewed before scraping because they did not all describe the same type of record. Some contained community projects, while others contained individual skills-development or boarding-school bursary beneficiaries.

Four PDFs were selected for this dataset:

1. `CDF-DUNDUMWEZI-2024.pdf`
2. `CDF-KALOMO-CENTRAL-2024.pdf`
3. `2025-DUNDUMWEZI-APPROVED-AND-NOT-APPROVED-PROJECTS.pdf`
4. `2025-KALOMO-CENTRAL-NOT-APPROVED-AND-APPROVED-PROJECTS.pdf`

These are the complete project lists for the two constituencies and two years. Separate approved-only and not-approved-only PDFs were not added because their records are already contained in the combined lists. Bursary registers were also excluded because their rows represent people rather than projects and contain personal identifiers that do not belong in the agreed CDF project schema.

In [ ]:
from pathlib import Path
import sys

repo_root = Path("..").resolve()
if not (repo_root / "scripts").exists():
    repo_root = Path(".").resolve()
sys.path.insert(0, str(repo_root / "scripts" / "scraping"))

import scrape_cdf_projects as scraper

print("Canonical project PDFs used:")
for filename in scraper.CANONICAL_PROJECT_PDFS:
    print(" -", filename)

## Step 2 - Extract the scanned tables with OCR

The selected PDFs are scanned documents, so ordinary PDF text extraction does not recover their tables reliably. `scrape_cdf_projects.py` renders each page at 300 DPI with `pypdfium2` and sends the page image to EasyOCR.

EasyOCR returns the detected text, a confidence score, and the coordinates of its bounding box. The coordinates are important because OCR detects individual table cells and text fragments, not complete project records. The scraper therefore saves the following raw fields:

- `text_line`
- `source_file_name`
- `page_number`
- `x_min`, `y_min`, `x_max`, `y_max`
- `ocr_confidence`

The OCR command used in Colab was:

```bash
python scripts/scraping/scrape_cdf_projects.py --pdf-dir /content/cdf_pdfs --gpu
```

The result was 3,804 OCR fragments saved to `data/raw/cdf_projects/raw_cdf_projects.csv`. A fragment is only part of a table row, so this number is not the number of projects.

In [ ]:
import pandas as pd

raw_path = repo_root / "data" / "raw" / "cdf_projects" / "raw_cdf_projects.csv"
raw = pd.read_csv(raw_path)

print("Raw OCR fragments:", len(raw))
print("Source PDFs:", raw["source_file_name"].nunique())
display(raw.head())
display(raw.groupby(["source_file_name", "page_number"]).size().rename("fragments").to_frame())

## Step 3 - Problems encountered

**The first OCR file was much larger than expected.** Running OCR across all 20 PDFs produced about 28,000 rows. This happened because one OCR text box was written as one CSV row and because the run included thousands of bursary beneficiary entries. It did not mean that the council had 28,000 projects.

**Some project documents overlap.** The site publishes complete combined lists as well as separate approved and not-approved versions. Treating all of them as independent sources would count the same applications more than once. The scraper was restricted to the four complete sources listed in Step 1.

**Reading OCR fragments in sequence merged table rows.** An early cleaner treated the OCR output as plain text. This produced short or mixed names such as ward names, descriptions, and headings being mistaken for projects. The corrected scraper retains bounding-box coordinates, and the cleaner uses those coordinates to rebuild the original table columns and rows.

**OCR spelling is not always exact.** Examples include `NOL APPROVED` for `NOT APPROVED`, `AIl Wards` for `All Wards`, and several misspellings of `Infrastructure`. The cleaner uses a small list of expected wards and sectors together with conservative fuzzy matching. It leaves a field as `N/A` when the source text is still too unclear to classify safely.

## Step 4 - Reconstruct and clean the project records

The 2024 and 2025 PDFs use different table layouts, so the cleaning script handles them separately.

For the 2024 tables, each recognised sector cell marks one project row. The bottom edge of the previous sector box and the current sector box are used to estimate the row boundaries. Text in the project-name and ward columns is then collected from the same vertical area.

For the 2025 tables, each `Approved` or `Not Approved` cell marks a project application. Repeated table headings are used to locate the project-name, description, sector, type, ward, comments, and reason columns. Nearby serial numbers provide an extra boundary when project names are tightly packed.

The cleaner also:

- removes headings, serial numbers, rejection reasons, stamps, and letterhead text from project names;
- standardises recognised ward, sector, and status values;
- keeps legitimate repeated applications instead of dropping them automatically;
- assigns sequential IDs from `CDF-0001`;
- records the council source page and cleaning date;
- checks project names for NRC and phone-number patterns before export; and
- writes the final file with a pipe (`|`) delimiter.

The published project tables do not provide reliable allocation or disbursement amounts for these rows, so `amount_allocated_zmw` and `amount_disbursed_zmw` are recorded as `N/A` rather than guessed.

In [ ]:
sys.path.insert(0, str(repo_root / "scripts" / "cleaning"))
import clean_cdf_projects as cleaner

print("Final column order:")
for column in cleaner.OUTPUT_COLUMNS:
    print(" -", column)

print("\nTo rebuild the processed file without repeating OCR:")
print("python scripts/cleaning/clean_cdf_projects.py")

## Step 5 - Load and inspect the final dataset

In [ ]:
processed_path = repo_root / "data" / "processed" / "db-unza26-csc4792-kalomo_town_council_cdf_projects.csv"
df = pd.read_csv(
    processed_path, sep="|", keep_default_na=False, dtype={"fiscal_year": str}
)
df.head(10)

In [ ]:
print("Total project records:", len(df))
print("\nRecords by year and status:")
display(df.groupby(["fiscal_year", "status"]).size().rename("records").to_frame())

print("Records by sector:")
display(df["sector"].value_counts().rename("records").to_frame())

print("Records by ward:")
display(df["ward"].value_counts().rename("records").to_frame())

## Step 6 - Validate the processed file

The checks below confirm the agreed schema, unique project IDs, recognised years and statuses, and the absence of personal identifier patterns. The expected result is 488 records: 53 from 2024 and 435 from 2025.

In [ ]:
import re

expected_columns = list(cleaner.OUTPUT_COLUMNS)
assert list(df.columns) == expected_columns
assert len(df) == 488
assert df["project_id"].is_unique
assert (df["project_name"] != "N/A").all()
assert set(df["fiscal_year"]) == {"2024", "2025"}
assert set(df["status"]) <= {"Approved", "Not Approved"}

nrc_pattern = re.compile(r"\b\d{4,7}\s*/\s*\d{1,3}(?:\s*/\s*\d)?\b")
phone_pattern = re.compile(r"\b0?9[567]\d{7}\b")
search_text = df.astype(str).agg(" ".join, axis=1)
assert not search_text.str.contains(nrc_pattern, regex=True).any()
assert not search_text.str.contains(phone_pattern, regex=True).any()

print("All validation checks passed.")
print("Missing ward values:", (df["ward"] == "N/A").sum())
print("Missing sector values:", (df["sector"] == "N/A").sum())

## Step 7 - Output format compliance

The final dataset follows the repository conventions:

- Filename: `db-unza26-csc4792-kalomo_town_council_cdf_projects.csv`
- Location: `data/processed/`
- Delimiter: pipe (`|`)
- Schema: the 10 CDF project columns documented in `docs/DATA_DICTIONARY.md`
- Unique identifier: `project_id`
- Provenance: `source_url` and `date_scraped` on every row

The raw OCR file remains under `data/raw/cdf_projects/` so the cleaning stage is reproducible without keeping the temporary PDF folder in the repository.

## Limitations

- OCR spelling errors remain in a small number of project names because the original PDFs are scanned images. Values were not manually rewritten unless the intended label could be matched confidently.
- One 2024 disaster-component record covers several health posts and does not identify one clear ward, so its ward is `N/A`.
- One 2025 sector cell remains unreadable after OCR and is kept as `N/A` rather than guessed.
- Allocation and disbursement amounts are `N/A` because the selected project tables do not provide reliable project-level figures in those columns.
- The dataset covers community projects only. Individual bursary beneficiaries were deliberately excluded because they require a different schema and introduce unnecessary personal information.
- The temporary source PDFs are not committed to the repository. The source URLs identify the council pages from which the documents were obtained.

See `docs/DATA_DICTIONARY.md` for the complete column descriptions.